# 01 — PVGIS data extraction

Downloads hourly PVGIS v5.3 (PVGIS-ERA5) simulations for seven Indonesian cities, 2019–2023, for a 1 kWp free-standing crystalline-silicon system (tilt 10°, azimuth 0°, system loss 14%).

**Note:** `P_kW` is PVGIS-simulated output, not measured plant output.

Output: `DATA_DIR/pvgis_indonesia_solar_dataset.csv` (306,768 rows), the input of notebook 02.

In [ ]:
# ============================================================
# CONFIGURATION — set DATA_DIR before running anything else
# ============================================================
import os

DATA_DIR = 'data'   # local run: folder that holds the input CSV and all outputs

try:                # Google Colab: mount Drive and point DATA_DIR to your Drive folder
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/TUGAS_AKHIR/Dataset'   # <-- change to your own Drive folder
except ImportError:
    pass

for sub in ['', 'tuning', 'stacking', 'multiseed', 'measured']:
    os.makedirs(os.path.join(DATA_DIR, sub), exist_ok=True)
print('DATA_DIR =', os.path.abspath(DATA_DIR))

In [1]:
!pip install requests pandas

In [2]:
import requests
import pandas as pd
import time
import os
from io import StringIO

# **Dataset**

**1. Konfigurasi kota**

mendefinisikan 7 kota yang dipilih untuk merepresentasikan distribusi geografis Indonesia dari barat ke timur. Masing-masing kota dilengkapi koordinat dan zona waktu (WIB, WITA, atau WIT) yang digunakan untuk konversi waktu UTC ke waktu lokal.

In [3]:
CITIES = [
    {"name": "Medan",      "lat":  3.5952,  "lon":  98.6722, "tz_offset": 7},
    {"name": "Pontianak",  "lat": -0.0263,  "lon": 109.3425, "tz_offset": 7},
    {"name": "Jakarta",    "lat": -6.2088,  "lon": 106.8456, "tz_offset": 7},
    {"name": "Surabaya",   "lat": -7.2575,  "lon": 112.7521, "tz_offset": 7},
    {"name": "Makassar",   "lat": -5.1477,  "lon": 119.4327, "tz_offset": 8},
    {"name": "Kupang",     "lat": -10.1772, "lon": 123.6070, "tz_offset": 8},
    {"name": "Jayapura",   "lat": -2.5337,  "lon": 140.7181, "tz_offset": 9},
]

print("Kota yang dikonfigurasi:")
for c in CITIES:
    print(f"  {c['name']:12} | lat={c['lat']:8.4f}, lon={c['lon']:9.4f} | UTC+{c['tz_offset']}")

Kota yang dikonfigurasi:
  Medan        | lat=  3.5952, lon=  98.6722 | UTC+7
  Pontianak    | lat= -0.0263, lon= 109.3425 | UTC+7
  Jakarta      | lat= -6.2088, lon= 106.8456 | UTC+7
  Surabaya     | lat= -7.2575, lon= 112.7521 | UTC+7
  Makassar     | lat= -5.1477, lon= 119.4327 | UTC+8
  Kupang       | lat=-10.1772, lon= 123.6070 | UTC+8
  Jayapura     | lat= -2.5337, lon= 140.7181 | UTC+9


**2. Konfigurasi Sistem PV & Rentang tahun**

Sistem PV dikonfigurasi dengan kapasitas 1 kWp sehingga output (kW) sama dengan daya spesifik per kWp terpasang dan mudah dibandingkan antar kota. Pengaturan lain: teknologi crystalline silicon (`crystSi`), pemasangan free-standing, tilt 10°, azimuth 0° (konvensi PVGIS, 0° = selatan), dan system loss 14%. Database radiasi yang dipakai adalah PVGIS-ERA5. Rentang tahun 2019–2023 dipilih karena merupakan lima tahun terakhir yang tersedia untuk PVGIS-ERA5 di PVGIS 5.3.

**Catatan:** `P_kW` adalah output simulasi model PV PVGIS yang dihitung dari irradiance, suhu, dan kecepatan angin yang sama dengan fitur input, bukan hasil pengukuran pembangkit.

In [4]:
PV_CONFIG = {
    "peakpower":      1,        # kapasitas nominal (kWp)
    "loss":          14,        # system loss (%) — nilai tipikal
    "mountingplace": "free",   # free-standing (bukan rooftop)
    "angle":         10,        # tilt angle rendah, cocok untuk tropis
    "aspect":         0,        # azimuth 0 = south-facing
    "pvtechchoice":  "crystSi", # crystalline silicon
}

START_YEAR = 2019
END_YEAR   = 2023

PVGIS_BASE = "https://re.jrc.ec.europa.eu/api/v5_3/seriescalc"

print(f"Konfigurasi PV : {PV_CONFIG}")
print(f"Rentang tahun  : {START_YEAR} – {END_YEAR}")
print(f"Estimasi data  : {(END_YEAR - START_YEAR + 1) * 8760 * len(CITIES):,} baris")

Konfigurasi PV : {'peakpower': 1, 'loss': 14, 'mountingplace': 'free', 'angle': 10, 'aspect': 0, 'pvtechchoice': 'crystSi'}
Rentang tahun  : 2019 – 2023
Estimasi data  : 306,600 baris


**3. Fetch Data PVGIS**

mengirim request ke API PVGIS untuk satu kota, lalu mem-parsing respons CSV-nya menjadi DataFrame. Kolom utama yang dihasilkan antara lain `irr_total`, `temp_air`, `wind_speed`, dan `P_kW` sebagai label target prediksi.

In [5]:
def fetch_pvgis_hourly(city: dict, start_year: int, end_year: int) -> pd.DataFrame:
    params = {
        "lat":           city["lat"],
        "lon":           city["lon"],
        "startyear":     start_year,
        "endyear":       end_year,
        "pvcalculation": 1,
        "peakpower":     PV_CONFIG["peakpower"],
        "loss":          PV_CONFIG["loss"],
        "mountingplace": PV_CONFIG["mountingplace"],
        "angle":         PV_CONFIG["angle"],
        "aspect":        PV_CONFIG["aspect"],
        "pvtechchoice":  PV_CONFIG["pvtechchoice"],
        "components":    1,
        "outputformat":  "csv",
        "browser":       0,
    }

    print(f"  Fetching {city['name']} ({city['lat']}, {city['lon']})...")

    try:
        resp = requests.get(PVGIS_BASE, params=params, timeout=120)
        resp.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"  Error fetching {city['name']}: {e}")
        return pd.DataFrame()

    lines      = resp.text.splitlines()
    data_start = None
    for i, line in enumerate(lines):
        if line.startswith("time,"):
            data_start = i
            break

    if data_start is None:
        print(f"  Tidak bisa parse data untuk {city['name']}")
        return pd.DataFrame()

    data_end = len(lines)
    for i in range(data_start + 1, len(lines)):
        if lines[i].strip() == "" or lines[i].startswith("-"):
            data_end = i
            break

    df = pd.read_csv(StringIO("\n".join(lines[data_start:data_end])))

    rename_map = {
        "time":   "datetime",
        "P":      "P_kW",
        "Gb(i)":  "irr_beam",
        "Gd(i)":  "irr_diffuse",
        "Gr(i)":  "irr_reflect",
        "H_sun":  "sun_height",
        "T2m":    "temp_air",
        "WS10m":  "wind_speed",
        "Int":    "flag",
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

    if "P_kW" in df.columns:
        df["P_kW"] = df["P_kW"] / 1000.0

    irr_cols = [c for c in ["irr_beam", "irr_diffuse", "irr_reflect"] if c in df.columns]
    if irr_cols:
        df["irr_total"] = df[irr_cols].sum(axis=1)

    if "datetime" in df.columns:
        df["datetime"]       = pd.to_datetime(df["datetime"], format="%Y%m%d:%H%M", errors="coerce")
        df["datetime_utc"]   = df["datetime"]
        tz_offset            = city.get("tz_offset", 7)
        df["datetime_local"] = df["datetime_utc"] + pd.Timedelta(hours=tz_offset)
        df["year"]           = df["datetime_local"].dt.year
        df["month"]          = df["datetime_local"].dt.month
        df["day"]            = df["datetime_local"].dt.day
        df["hour"]           = df["datetime_local"].dt.hour
        df["season"]         = df["month"].map({
            11: "wet", 12: "wet", 1: "wet", 2: "wet", 3: "wet",
            4:  "dry",  5: "dry", 6: "dry", 7: "dry",
            8:  "dry",  9: "dry", 10: "dry"
        })

    df["city"]      = city["name"]
    df["latitude"]  = city["lat"]
    df["longitude"] = city["lon"]
    df["tz_offset"] = city.get("tz_offset", 7)

    print(f"  {city['name']}: {len(df):,} baris berhasil diambil")
    return df

print("Fungsi fetch_pvgis_hourly siap digunakan.")

Fungsi fetch_pvgis_hourly siap digunakan.


eksekusi fetch data untuk semua kota
<br>Fungsi fetch dipanggil satu per satu untuk setiap kota

In [6]:
print("=" * 60)
print("Memulai pengambilan data PVGIS...")
print(f"Kota  : {', '.join(c['name'] for c in CITIES)}")
print(f"Tahun : {START_YEAR} – {END_YEAR}")
print("=" * 60)

all_dfs = []

for city in CITIES:
    df_city = fetch_pvgis_hourly(city, START_YEAR, END_YEAR)
    if not df_city.empty:
        all_dfs.append(df_city)
    time.sleep(2)

if not all_dfs:
    print("Tidak ada data yang berhasil diambil. Cek koneksi internet.")
else:
    print(f"\nBerhasil mengambil data dari {len(all_dfs)} kota.")

Memulai pengambilan data PVGIS...
Kota  : Medan, Pontianak, Jakarta, Surabaya, Makassar, Kupang, Jayapura
Tahun : 2019 – 2023
  Fetching Medan (3.5952, 98.6722)...
  Medan: 43,824 baris berhasil diambil
  Fetching Pontianak (-0.0263, 109.3425)...
  Pontianak: 43,824 baris berhasil diambil
  Fetching Jakarta (-6.2088, 106.8456)...
  Jakarta: 43,824 baris berhasil diambil
  Fetching Surabaya (-7.2575, 112.7521)...
  Surabaya: 43,824 baris berhasil diambil
  Fetching Makassar (-5.1477, 119.4327)...
  Makassar: 43,824 baris berhasil diambil
  Fetching Kupang (-10.1772, 123.607)...
  Kupang: 43,824 baris berhasil diambil
  Fetching Jayapura (-2.5337, 140.7181)...
  Jayapura: 43,824 baris berhasil diambil

Berhasil mengambil data dari 7 kota.


DataFrame dari semua kota digabungkan menjadi satu dataset. Urutan kolom disusun secara logis mulai dari informasi temporal, spasial, meteorologis, hingga label target `P_kW` di bagian akhir.

In [7]:
combined = pd.concat(all_dfs, ignore_index=True)

col_order = [
    "datetime_utc", "datetime_local",
    "city", "latitude", "longitude", "tz_offset",
    "year", "month", "day", "hour", "season",
    "irr_total", "irr_beam", "irr_diffuse", "irr_reflect",
    "temp_air", "wind_speed", "sun_height",
    "P_kW",
    "flag",
]
col_order = [c for c in col_order if c in combined.columns]
combined  = combined[col_order]

print(f"Shape dataset gabungan : {combined.shape}")
print(f"Kolom                  : {list(combined.columns)}")
combined.head()

Shape dataset gabungan : (306768, 20)
Kolom                  : ['datetime_utc', 'datetime_local', 'city', 'latitude', 'longitude', 'tz_offset', 'year', 'month', 'day', 'hour', 'season', 'irr_total', 'irr_beam', 'irr_diffuse', 'irr_reflect', 'temp_air', 'wind_speed', 'sun_height', 'P_kW', 'flag']


,datetime_utc,datetime_local,city,latitude,longitude,tz_offset,year,month,day,hour,season,irr_total,irr_beam,irr_diffuse,irr_reflect,temp_air,wind_speed,sun_height,P_kW,flag
0,2019-01-01 00:30:00,2019-01-01 07:30:00,Medan,3.5952,98.6722,7,2019,1,1,7,wet,146.09,65.76,80.14,0.19,23.36,1.66,12.60,0.09764,0.0
1,2019-01-01 01:30:00,2019-01-01 08:30:00,Medan,3.5952,98.6722,7,2019,1,1,8,wet,353.86,208.08,145.29,0.49,24.46,1.72,26.05,0.27097,0.0
2,2019-01-01 02:30:00,2019-01-01 09:30:00,Medan,3.5952,98.6722,7,2019,1,1,9,wet,543.31,337.46,205.08,0.77,25.98,2.28,38.95,0.42113,0.0
3,2019-01-01 03:30:00,2019-01-01 10:30:00,Medan,3.5952,98.6722,7,2019,1,1,10,wet,523.74,236.94,286.04,0.76,27.04,2.62,50.68,0.40448,0.0
4,2019-01-01 04:30:00,2019-01-01 11:30:00,Medan,3.5952,98.6722,7,2019,1,1,11,wet,661.79,352.14,308.69,0.96,27.54,2.14,59.78,0.50409,0.0


In [8]:
out_path = os.path.join(DATA_DIR, "pvgis_indonesia_solar_dataset.csv")
combined.to_csv(out_path, index=False)

print("=" * 60)
print("SELESAI — Ringkasan Dataset")
print("=" * 60)
print(f"Total baris        : {len(combined):,}")
print(f"Total kolom        : {len(combined.columns)}")
print(f"Kota tercakup      : {combined['city'].nunique()}")
print(f"Rentang waktu UTC  : {combined['datetime_utc'].min()}  s/d  {combined['datetime_utc'].max()}")
print(f"\nLabel (P_kW) — satuan kW, belum dinormalisasi:")
print(f"  Min  : {combined['P_kW'].min():.4f} kW")
print(f"  Max  : {combined['P_kW'].max():.4f} kW")
print(f"  Mean : {combined['P_kW'].mean():.4f} kW")
print(f"\nFile tersimpan di  : {os.path.abspath(out_path)}")
print("=" * 60)

SELESAI — Ringkasan Dataset
Total baris        : 306,768
Total kolom        : 20
Kota tercakup      : 7
Rentang waktu UTC  : 2019-01-01 00:30:00  s/d  2023-12-31 23:30:00

Label (P_kW) — satuan kW, belum dinormalisasi:
  Min  : 0.0000 kW
  Max  : 0.7822 kW
  Mean : 0.1538 kW

File tersimpan di  : /content/pvgis_indonesia_solar_dataset.csv


In [9]:
# Cek database radiasi & pengaturan sistem dari header respons PVGIS (tidak mengubah dataset)
params = {
    "lat": CITIES[0]["lat"], "lon": CITIES[0]["lon"],
    "startyear": START_YEAR, "endyear": START_YEAR,      # 1 tahun saja agar cepat
    "pvcalculation": 1,
    "peakpower": PV_CONFIG["peakpower"], "loss": PV_CONFIG["loss"],
    "mountingplace": PV_CONFIG["mountingplace"], "angle": PV_CONFIG["angle"],
    "aspect": PV_CONFIG["aspect"], "pvtechchoice": PV_CONFIG["pvtechchoice"],
    "components": 1, "outputformat": "csv", "browser": 0,
}
resp = requests.get(PVGIS_BASE, params=params, timeout=120)
resp.raise_for_status()
for line in resp.text.splitlines():
    if line.startswith("time,"):
        break
    print(line)

Latitude (decimal degrees):	3.595
Longitude (decimal degrees):	98.672
Elevation (m):	19
Radiation database:	PVGIS-ERA5


Slope: 10 deg. 
Azimuth: 0 deg. 
Nominal power of the PV system (c-Si) (kWp):	1.0
System losses (%):	14.0
